In [1]:
import sys
sys.path.insert(0, "..")

import os

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from src import config as C

In [2]:
df = pd.read_csv("../data/processed/customer_churn_clean.csv")

In [3]:
overall = df["Churn"].mean()
plt.rcParams.update({
    "font.size": 11, 
    "axes.spines.top": False, 
    "axes.spines.right": False,
    "axes.edgecolor": C.GRID, 
    "axes.labelcolor": C.MUTED, 
    "xtick.color": C.MUTED,
    "ytick.color": C.MUTED, 
    "figure.facecolor": "white"
})

In [4]:
def header(fig, title, q):
    fig.text(0.01, 0.965, title, fontsize=13, fontweight="bold", color=C.INK, va="top")
    fig.text(0.01, 0.905, q, color=C.MUTED, fontsize=10, va="top")


def save(fig, name): 
    os.makedirs("../reports/figures", exist_ok=True)
    fig.tight_layout(rect=[0, 0, 1, 0.86])
    fig.savefig(f"../reports/figures/{name}.png", dpi=150)
    plt.close(fig)


def rate_chart(groups, order, title, question, name, xlabel, rotate=0):
    t = df.groupby(groups, observed=True)["Churn"].agg(["mean", "count"]).reindex(order)
    fig, ax = plt.subplots(figsize=(8, 4.6))
    bars = ax.bar(range(len(t)), t["mean"] * 100, color=C.BLUE, width=0.62)
    
    ax.axhline(overall * 100, color=C.ORANGE, lw=2, ls="--")
    ax.text(len(t) - 0.5, overall * 100 + 0.8, f"Overall {overall*100:.1f}%", color=C.MUTED, ha="right", fontsize=10)
    
    for b, v in zip(bars, t["mean"] * 100):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.6, f"{v:.0f}%", ha="center", color=C.INK, fontsize=10)
    
    ax.set_xticks(range(len(t)))
    ax.set_xticklabels(order, rotation=rotate, ha="right" if rotate else "center")
    ax.set_ylabel("Churn rate (%)")
    ax.set_xlabel(xlabel); ax.yaxis.grid(True, color=C.GRID)
    ax.set_axisbelow(True)
    
    header(fig, title, question)
    save(fig, name)
    
    return t

In [5]:
# 1 distribution
vc = df["Churn"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4.2))
b = ax.bar(["Retained", "Churned"], vc.values, color=[C.BLUE, C.ORANGE], width=0.55)

for r, v in zip(b, vc.values): 
    ax.text(r.get_x() + r.get_width()/2, v+40, f"{v:,}  ({v/len(df)*100:.1f}%)", ha="center", color=C.INK)

ax.set_ylabel("Customers")
ax.yaxis.grid(True, color=C.GRID)
ax.set_axisbelow(True)

header(fig, "Churn distribution", "How imbalanced is the target? (decides how we evaluate models)")
save(fig, "01_churn_distribution")


In [6]:
res = {}

res["subscription"] = rate_chart(
    "Subscription_Type", 
    ["Basic","Standard","Premium"], 
    "Churn by subscription type",
    "Which plan loses the most customers?", 
    "02_churn_by_subscription", 
    "Subscription type"
)

res["contract"] = rate_chart(
    "Contract_Type", 
    ["Month-to-Month","One Year","Two Year"], 
    "Churn by contract type",
    "Do longer contracts protect against churn?", 
    "03_churn_by_contract", 
    "Contract type"
)

loc_order = df.groupby("Location")["Churn"].mean().sort_values(ascending=False).index.tolist()

res["location"] = rate_chart(
    "Location", 
    loc_order, 
    "Churn by location",
    "Are some cities riskier than others?", 
    "04_churn_by_location", 
    "Location", 
    rotate=30
)

df["Tenure_Band"] = pd.cut(df["Tenure_Months"], [0,6,12,24,36,72], labels=["0-6","7-12","13-24","25-36","37-72"])

res["tenure"] = rate_chart(
    "Tenure_Band", 
    ["0-6","7-12","13-24","25-36","37-72"], 
    "Churn vs tenure",
    "Do newer customers churn more often?", 
    "05_churn_vs_tenure", 
    "Tenure (months)"
)

df["Charge_Band"] = pd.qcut(df["Monthly_Charges"], 5, labels=["Lowest 20%","Low","Mid","High","Highest 20%"])

res["charges"] = rate_chart(
    "Charge_Band", 
    ["Lowest 20%","Low","Mid","High","Highest 20%"], 
    "Churn vs monthly charges",
    "Are high-paying customers more likely to leave?", 
    "06_churn_vs_monthly_charges", 
    "Monthly charges (quintile)"
)

df["Ticket_Band"] = pd.cut(df["Support_Tickets"], [-1,0,2,4,100], labels=["0","1-2","3-4","5+"])

res["tickets"] = rate_chart(
    "Ticket_Band", 
    ["0","1-2","3-4","5+"], 
    "Churn vs support tickets",
    "Does heavy support usage signal dissatisfaction?", 
    "07_churn_vs_support_tickets", 
    "Support tickets"
)

res["satisfaction"] = rate_chart(
    "Customer_Satisfaction", 
    [1,2,3,4,5], 
    "Churn vs customer satisfaction",
    "Does low satisfaction go with churn?", 
    "08_churn_vs_satisfaction", 
    "Satisfaction score (1-5)"
)

df["Login_Band"] = pd.cut(df["Login_Frequency"], [-1,5,10,15,20,100], labels=["0-5","6-10","11-15","16-20","21+"])

res["logins"] = rate_chart(
    "Login_Band", 
    ["0-5","6-10","11-15","16-20","21+"], 
    "Churn vs login frequency",
    "Are inactive customers more likely to churn?", 
    "09_churn_vs_login_frequency", 
    "Logins per month"
)

df["Complaint_Band"] = pd.cut(df["Complaints"], [-1,0,1,2,100], labels=["0","1","2","3+"])

res["complaints"] = rate_chart(
    "Complaint_Band", 
    ["0","1","2","3+"], 
    "Churn vs complaints",
    "Do multiple complaints raise churn?", 
    "10_churn_vs_complaints", 
    "Complaints"
)

In [7]:
# correlation heatmap
num = df.select_dtypes("number").drop(columns=[]).corr()

cmap = LinearSegmentedColormap.from_list("div", [C.BLUE, C.WHITE, C.ORANGE])

fig, ax = plt.subplots(figsize=(9.5, 7.8))
im = ax.imshow(num, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(len(num))); ax.set_xticklabels(num.columns, rotation=45, ha="right")
ax.set_yticks(range(len(num))); ax.set_yticklabels(num.columns)

for i in range(len(num)):
    for j in range(len(num)):
        v = num.iat[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if abs(v) > .6 else C.INK)

ax.set_title("Correlation heatmap", loc="left", fontsize=13, fontweight="bold", color=C.INK, pad=22)

fig.text(
    0.01, 0.945, 
    "Which numeric features move together with Churn?  (blue = negative, orange = positive)", 
    color=C.MUTED, 
    fontsize=10
)
fig.colorbar(im, shrink=0.7)

save(fig, "11_correlation_heatmap")

pd.set_option("display.width", 200)

for k, t in res.items():
    print(f"\n== {k} =="); print((t.assign(rate=(t['mean']*100).round(1))[["rate","count"]]).T.to_string())


== subscription ==
Subscription_Type   Basic  Standard  Premium
rate                 14.0      16.1     18.8
count              3962.0    3841.0   2197.0

== contract ==
Contract_Type  Month-to-Month  One Year  Two Year
rate                     22.4      10.1       4.3
count                  5465.0    2877.0    1658.0

== location ==
Location  Kolkata  Mumbai  Chennai   Delhi    Pune  Hyderabad  Bengaluru  Ahmedabad
rate         17.7    16.4     16.1    16.1    15.8       15.6       15.1       13.6
count      1002.0  1568.0   1062.0  1673.0  1185.0     1175.0     1557.0      778.0

== tenure ==
Tenure_Band    0-6    7-12   13-24   25-36   37-72
rate          25.4    21.1    19.1    15.6     9.4
count        559.0  1180.0  2869.0  2252.0  3140.0

== charges ==
Charge_Band  Lowest 20%     Low     Mid    High  Highest 20%
rate               13.8    14.0    15.8    16.6         19.0
count            2000.0  2000.0  2000.0  2000.0       2000.0

== tickets ==
Ticket_Band       0     1-2    

In [8]:
print("\nCorrelation with Churn:")
print(num["Churn"].drop("Churn").sort_values().round(3).to_string())

print("\nMean by churn:")
print(
    df.groupby("Churn")[["Tenure_Months","Monthly_Charges","Support_Tickets","Complaints","Login_Frequency","Last_Login_Days","Usage_Hours","Customer_Satisfaction"]].mean().round(2).T.to_string()
)


Correlation with Churn:
Customer_Satisfaction   -0.175
Tenure_Months           -0.144
Login_Frequency         -0.097
Total_Charges           -0.077
Discount_Received       -0.055
Usage_Hours             -0.022
Age                      0.001
Monthly_Charges          0.053
Last_Login_Days          0.113
Support_Tickets          0.115
Complaints               0.194

Mean by churn:
Churn                       0       1
Tenure_Months           31.13   23.93
Monthly_Charges        575.54  615.02
Support_Tickets          1.42    1.81
Complaints               0.68    1.17
Login_Frequency         13.07   12.05
Last_Login_Days          8.44   11.16
Usage_Hours             21.77   20.94
Customer_Satisfaction    3.76    3.41
